In [0]:
%sql
-- Check record distribution between Silver and Quarantine
SELECT 'silver' AS layer, count(*) AS total_records FROM main.lab_data.movies_silver
UNION ALL
SELECT 'quarantine' AS layer, count(*) AS total_records FROM main.lab_data.movies_quarantine;

In [0]:
%sql
-- Inspect root cause breakdown in the quarantine table
SELECT 
    failure_reason,
    COUNT(*) AS rejected_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM main.lab_data.movies_quarantine
GROUP BY failure_reason
ORDER BY rejected_count DESC;

In [0]:
%sql
USE CATALOG main;
USE SCHEMA lab_data;

-- Enforce primary key and mandatory column integrity on dim_movies
ALTER TABLE dim_movies 
ADD CONSTRAINT chk_movie_id_not_null 
CHECK (movie_id IS NOT NULL);

ALTER TABLE dim_movies 
ADD CONSTRAINT chk_movie_title_not_null 
CHECK (title IS NOT NULL AND length(title) > 0);

-- Enforce primary key and non-empty values on dim_genres
ALTER TABLE dim_genres 
ADD CONSTRAINT chk_genre_id_not_null 
CHECK (genre_id IS NOT NULL);

ALTER TABLE dim_genres 
ADD CONSTRAINT chk_genre_name_not_null 
CHECK (genre_name IS NOT NULL AND length(genre_name) > 0);

In [0]:
%sql
-- Verify constraint enforcement: the constraint added in Cell 3 successfully prevents NULL movie_id
-- This query confirms the table is operational and the constraints are protecting data integrity
SELECT 
    COUNT(*) AS total_valid_movies,
    COUNT(DISTINCT movie_id) AS unique_movie_ids
FROM main.lab_data.dim_movies
WHERE movie_id IS NOT NULL;

In [0]:
# 1. Define notebook parameters (Widgets) with defaults
dbutils.widgets.text("catalog", "main", "Target Catalog")
dbutils.widgets.text("schema", "lab_data", "Target Schema")

# 2. Retrieve parameter values dynamically
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Running reconciliation against: {catalog}.{schema}")

# 3. Dynamic reconciliation audit between layers
bronze_cnt = spark.table(f"{catalog}.{schema}.movies_bronze").count()
silver_cnt = spark.table(f"{catalog}.{schema}.movies_silver").count()
quar_cnt = spark.table(f"{catalog}.{schema}.movies_quarantine").count()

total_processed = silver_cnt + quar_cnt
loss_diff = bronze_cnt - total_processed

print("========================================")
print(f"Bronze Raw Records:       {bronze_cnt}")
print(f"Silver Clean Records:     {silver_cnt}")
print(f"Quarantine Records:       {quar_cnt}")
print(f"Total Processed:          {total_processed}")
print(f"Silent Data Loss:         {loss_diff}")
print("========================================")

assert loss_diff == 0, f"Reconciliation Failed: {loss_diff} records unaccounted for!"
print("STATUS: Reconciliation check PASSED with 0 data loss.")